# 07 — Evaluate All Models on Adverse Conditions

Evaluates all 4 models on every adverse condition split discovered during
data preparation (loaded dynamically from `configs/conditions.json`).

Robustness metrics (mAP drop, retention %) are computed relative to the
clear-day baseline loaded from `results/clear_day/scores.json`.

**Prerequisites:** notebook 06 must have completed successfully.

In [2]:
from pathlib import Path
import json, torch
from dotenv import load_dotenv
from src.eval_utils import (
    compute_map, compute_precision_recall, compute_per_class_ap,
    compute_robustness_metrics, compute_per_class_robustness, evaluate_rfdetr,
)
from src.data_utils import CLASS_NAMES
from src.train_utils import setup_logging

DRIVE_ROOT    = Path('/content/drive/MyDrive/FON/master_rad')
CONFIGS_DIR   = Path('/content/data_prepared/configs')
COCO_DATA_DIR = Path('/content/data_prepared/coco')
CHECKPOINTS   = DRIVE_ROOT / 'checkpoints'
RESULTS_ROOT  = DRIVE_ROOT / 'results'

load_dotenv(DRIVE_ROOT / '.env')
logger = setup_logging()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

conditions = json.loads((CONFIGS_DIR / 'conditions.json').read_text())
ADVERSE_CONDITIONS = conditions['adverse']
print(f'Loaded {len(ADVERSE_CONDITIONS)} adverse conditions:', ADVERSE_CONDITIONS)

clear_scores    = json.loads((RESULTS_ROOT / 'clear_day' / 'scores.json').read_text())
clear_per_class = json.loads((RESULTS_ROOT / 'clear_day' / 'per_class_ap.json').read_text())
print('Loaded clear-day baselines for:', list(clear_scores.keys()))

Loaded 12 adverse conditions: ['clear_dawn_dusk', 'clear_night', 'overcast_dawn_dusk', 'overcast_daytime', 'partly_cloudy_dawn_dusk', 'partly_cloudy_daytime', 'rainy_dawn_dusk', 'rainy_daytime', 'rainy_night', 'snowy_dawn_dusk', 'snowy_daytime', 'snowy_night']
Loaded clear-day baselines for: ['yolov11', 'yolov12', 'rtdetr', 'rfdetr']


In [3]:
from ultralytics import YOLO, RTDETR
from rfdetr import RFDETRMedium

yolov11 = YOLO(str(CHECKPOINTS / 'yolov11/run/weights/best.pt'))
yolov12 = YOLO(str(CHECKPOINTS / 'yolov12/run/weights/best.pt'))
rtdetr  = RTDETR(str(CHECKPOINTS / 'rtdetr/run/weights/best.pt'))
rfdetr  = RFDETRMedium(pretrain_weights=str(CHECKPOINTS / 'rfdetr/checkpoint_best_total.pth'))
rfdetr.optimize_for_inference()

[2026-05-10 19:01:42] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-10 19:01:42] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-10 19:01:43] [WARNING] rf-detr - Checkpoint has 10 classes but model is configured for 90. Using checkpoint class count (10). Pass num_classes=10 to suppress this warning.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


In [4]:
for condition in ADVERSE_CONDITIONS:
    RESULTS_DIR = RESULTS_ROOT / condition
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    scores = {}
    robustness = {}
    per_class_ap = {}

    for name, model in [('yolov11', yolov11), ('yolov12', yolov12), ('rtdetr', rtdetr)]:
        logger.info(f'Evaluating {name} on {condition}...')
        results = model.val(
            data=str(CONFIGS_DIR / 'dataset.yaml'),
            split=f'test_{condition}',
            device=DEVICE,
        )
        m = {**compute_map(results), **compute_precision_recall(results)}
        scores[name] = m
        per_class_ap[name] = compute_per_class_ap(results, CLASS_NAMES)
        rob = compute_robustness_metrics(
            clear_map=clear_scores[name]['map50'],
            adverse_map=m['map50'],
        )
        robustness[name] = rob
        logger.info(
            f'{name} {condition}: mAP50={m["map50"]:.4f}  '
            f'drop={rob["map_drop"]:.4f}  retention={rob["retention_pct"]:.1f}%'
        )

    logger.info(f'Evaluating rfdetr on {condition}...')
    rfdetr_result = evaluate_rfdetr(rfdetr, COCO_DATA_DIR, split=condition)
    scores['rfdetr'] = compute_map(rfdetr_result)
    per_class_ap['rfdetr'] = rfdetr_result.get('per_class_ap', {})
    robustness['rfdetr'] = compute_robustness_metrics(
        clear_map=clear_scores['rfdetr']['map50'],
        adverse_map=scores['rfdetr']['map50'],
    )
    logger.info(
        f"rfdetr {condition}: mAP50={scores['rfdetr']['map50']:.4f}  "
        f"drop={robustness['rfdetr']['map_drop']:.4f}  "
        f"retention={robustness['rfdetr']['retention_pct']:.1f}%"
    )

    # Per-class robustness for all classes (thesis highlights person and bike)
    per_class_robustness = {
        name: compute_per_class_robustness(clear_per_class[name], per_class_ap[name])
        for name in scores
    }

    (RESULTS_DIR / 'scores.json').write_text(json.dumps(scores, indent=2))
    (RESULTS_DIR / 'robustness.json').write_text(json.dumps(robustness, indent=2))
    (RESULTS_DIR / 'per_class_ap.json').write_text(json.dumps(per_class_ap, indent=2))
    (RESULTS_DIR / 'per_class_robustness.json').write_text(json.dumps(per_class_robustness, indent=2))
    print(f'Saved {condition} results to {RESULTS_DIR}')

19:01:48 | INFO     | Evaluating yolov11 on clear_dawn_dusk...


Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11m summary (fused): 126 layers, 20,037,742 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2131.8±522.3 MB/s, size: 57.8 KB)
val: Scanning /content/data_prepared/yolo/clear_dawn_dusk/labels.cache... 2311 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2311/2311 403.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 145/145 9.4it/s 15.4s<0.2s
                   all       2311      42162      0.707      0.465      0.502      0.282
                   car       2283      25703      0.808      0.723      0.784      0.484
                person        592       1937      0.712      0.514      0.582      0.287
          traffic sign       1848       7647       0.71      0.589      0.641       0.35
         traffic light       1105       5096      0.675      0.553      0.579      0.22

19:02:07 | INFO     | yolov11 clear_dawn_dusk: mAP50=0.5025  drop=0.0348  retention=93.5%
19:02:07 | INFO     | Evaluating yolov12 on clear_dawn_dusk...


Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv12m summary (fused): 169 layers, 20,112,622 parameters, 0 gradients, 67.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1880.6±593.2 MB/s, size: 49.1 KB)
val: Scanning /content/data_prepared/yolo/clear_dawn_dusk/labels.cache... 2311 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2311/2311 969.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 145/145 8.8it/s 16.5s<0.2s
                   all       2311      42162      0.713      0.449      0.495      0.277
                   car       2283      25703      0.824       0.71      0.778      0.481
                person        592       1937      0.711      0.496      0.568      0.279
          traffic sign       1848       7647       0.72      0.576       0.64      0.347
         traffic light       1105       5096      0.709      0.529      0.583      0.2

19:02:28 | INFO     | yolov12 clear_dawn_dusk: mAP50=0.4945  drop=0.0221  retention=95.7%
19:02:28 | INFO     | Evaluating rtdetr on clear_dawn_dusk...


...

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
rt-detr-l summary: 310 layers, 32,004,290 parameters, 0 gradients, 103.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1101.8±425.4 MB/s, size: 44.6 KB)
val: Scanning /content/data_prepared/yolo/snowy_night/labels.cache... 2522 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2522/2522 1.2Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 158/158 6.6it/s 24.0s0.1s
                   all       2522      41684      0.614      0.406      0.417      0.215
                   car       2487      22137      0.763       0.68       0.74      0.408
                person        649       1934      0.676      0.497      0.552      0.268
          traffic sign       2034       7884      0.591      0.713      0.656      0.335
         traffic light       1828       8714      0.614      0.522      0.487      0.151
          

19:55:59 | INFO     | rtdetr snowy_night: mAP50=0.4167  drop=0.1285  retention=76.4%
19:55:59 | INFO     | Evaluating rfdetr on snowy_night...


loading annotations into memory...
Done (t=0.15s)
creating index...
index created!
Loading and preparing results...
DONE (t=2.22s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=60.81s).
Accumulating evaluation results...
DONE (t=7.07s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.247
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.460
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.227
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.073
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.314
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.507
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.245
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.407
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

19:59:03 | INFO     | rfdetr snowy_night: mAP50=0.4597  drop=0.1037  retention=81.6%


Saved snowy_night results to /content/drive/MyDrive/FON/master_rad/results/snowy_night
